In [ ]:
# import all dependencies

import kagglehub
import pandas as pd
import glob
import os
import gc

In [ ]:
# ======================================================
# I - Data import part
# ======================================================

# Download dataset

path = kagglehub.dataset_download("elemento/nyc-yellow-taxi-trip-data")

In [3]:
print("Path to dataset files:", path)

Path to dataset files: C:\Users\Anya\.cache\kagglehub\datasets\elemento\nyc-yellow-taxi-trip-data\versions\2


In [4]:
# ==================================
# 1. Parse in chunks to save memory
# ==================================

dtypes = {
    'VendorID': 'int8',
    'passenger_count': 'int8',
    'trip_distance': 'float32',
    'pickup_longitude': 'float32',
    'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32',
    'dropoff_latitude': 'float32',
    'fare_amount': 'float32',
    'tip_amount': 'float32',
    'total_amount': 'float32'
}

parse_dates = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']

csv_files = glob.glob(f"{path}/yellow_tripdata_2015-*.csv") + \
            glob.glob(f"{path}/yellow_tripdata_2016-*.csv")
csv_files.sort()

# Define the folder path
folder_path = 'C:/Users/Anya/master_thesis/tmp'

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

# Process in chunks, save to temporary parquet files
chunk_size = 1_000_000
temp_files = []

for file_idx, file in enumerate(csv_files):
    print(f"Processing {file}...")

    for chunk_idx, chunk in enumerate(pd.read_csv(file,
                                                    dtype=dtypes,
                                                    parse_dates=parse_dates,
                                                    chunksize=chunk_size)):
        # Filter chunk to New York pickup coordinates square only, with valid trip distance and fare amount
        chunk = chunk[
            (chunk['pickup_longitude'].between(-74.05, -73.75)) &
            (chunk['pickup_latitude'].between(40.63, 40.86)) &
            (chunk['trip_distance'] > 0) &
            (chunk['fare_amount'] > 0)
        ]

        if len(chunk) > 0:
            # Save to temporary parquet file
            temp_file = f"{folder_path}/chunk_{file_idx}_{chunk_idx}.parquet"
            chunk.to_parquet(temp_file)
            temp_files.append(temp_file)
            print(f"Chunk {chunk_idx}: {len(chunk)} rows saved")

Processing C:\Users\Anya\.cache\kagglehub\datasets\elemento\nyc-yellow-taxi-trip-data\versions\2\yellow_tripdata_2015-01.csv...
Chunk 0: 974671 rows saved
Chunk 1: 974511 rows saved
Chunk 2: 975245 rows saved
Chunk 3: 974554 rows saved
Chunk 4: 974659 rows saved
Chunk 5: 974725 rows saved
Chunk 6: 975469 rows saved
Chunk 7: 974684 rows saved
Chunk 8: 974800 rows saved
Chunk 9: 974620 rows saved
Chunk 10: 974748 rows saved
Chunk 11: 974940 rows saved
Chunk 12: 730002 rows saved
Processing C:\Users\Anya\.cache\kagglehub\datasets\elemento\nyc-yellow-taxi-trip-data\versions\2\yellow_tripdata_2016-01.csv...
Chunk 0: 976420 rows saved
Chunk 1: 968349 rows saved
Chunk 2: 979065 rows saved
Chunk 3: 979334 rows saved
Chunk 4: 978668 rows saved
Chunk 5: 979787 rows saved
Chunk 6: 977476 rows saved
Chunk 7: 978073 rows saved
Chunk 8: 977411 rows saved
Chunk 9: 981532 rows saved
Chunk 10: 894799 rows saved
Processing C:\Users\Anya\.cache\kagglehub\datasets\elemento\nyc-yellow-taxi-trip-data\versio

In [ ]:
# ==================================================================
# 2. Delete temporary data and clean memory with garbage collection
# ==================================================================

del temp_files
del chunk
del csv_files

gc.collect()